In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/protein_sales.parquet')
df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
print("Memory:", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

Shape: (9315330, 12)
Memory: 1083.7 MB
Date range: 2013-01-01 to 2017-08-15


In [2]:
# How many rows in each period?
rows_2013_2014 = (df["date"] < "2015-01-01").sum()
rows_2015_plus = (df["date"] >= "2015-01-01").sum()

print(f"2013-2014 rows: {rows_2013_2014:,}")
print(f"2015+ rows:     {rows_2015_plus:,}")
print(f"Retained:       {rows_2015_plus / len(df) * 100:.1f}%")

# Days of history retained
days_retained = df[df["date"] >= "2015-01-01"]["date"].nunique()
print(f"\nDays of history: {days_retained} ({days_retained/365:.1f} years)")

2013-2014 rows: 3,380,802
2015+ rows:     5,934,528
Retained:       63.7%

Days of history: 956 (2.6 years)


In [3]:
# 1. Restrict to the period with complete promotional data
df = df[df["date"] >= "2015-01-01"].copy()
print("After date cut:", df.shape)

# 2. Inspect the negatives before deciding
negatives = df[df["unit_sales"] < 0]
print(f"\nNegative rows: {len(negatives)}")
print(negatives[["date", "store_nbr", "item_nbr", "family", "unit_sales"]].head(10))
print("\nNegative sales by family:")
print(negatives["family"].value_counts())

After date cut: (5934528, 12)

Negative rows: 298
              date  store_nbr  item_nbr   family  unit_sales
3049003 2015-02-16         39   1239903  POULTRY      -1.211
3054101 2015-03-12          6   1085246    MEATS      -1.314
3158668 2015-02-17         20    584129    MEATS      -0.082
3162170 2015-02-18          2    108831  POULTRY      -2.945
3285303 2015-02-11         48    584188    MEATS      -0.156
3378309 2015-01-06          4    108831  POULTRY      -2.283
3383694 2015-01-07          6    584026    MEATS      -0.068
3432715 2015-03-26         25    507969  POULTRY      -3.678
3438470 2015-03-27         26    582864    MEATS     -21.620
3441444 2015-04-16          1    584026    MEATS      -0.232

Negative sales by family:
family
POULTRY           112
MEATS              87
DELI               55
PREPARED FOODS     33
SEAFOOD            11
Name: count, dtype: int64


In [4]:
# Clip negatives to zero — demand can't be negative
df["unit_sales"] = df["unit_sales"].clip(lower=0)

print("Negatives remaining:", (df["unit_sales"] < 0).sum())
print("Min unit_sales:", df["unit_sales"].min())

Negatives remaining: 0
Min unit_sales: 0.0


In [5]:
print("onpromotion dtype:", df["onpromotion"].dtype)
print("Unique values:", df["onpromotion"].unique())
print("Nulls:", df["onpromotion"].isna().sum())
print("\nMax item_nbr:", df["item_nbr"].max())
print("Max class:", df["class"].max())

onpromotion dtype: str
Unique values: <ArrowStringArray>
['False', 'True']
Length: 2, dtype: str
Nulls: 0

Max item_nbr: 2081175
Max class: 2986


In [10]:
df = pd.read_parquet('../data/processed/protein_sales.parquet')
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2015-01-01"].copy()
df["unit_sales"] = df["unit_sales"].clip(lower=0)
print("Reloaded:", df.shape, "| promo values:", df["onpromotion"].unique())

Reloaded: (5934528, 12) | promo values: <ArrowStringArray>
['False', 'True']
Length: 2, dtype: str


In [11]:
# Downcast columns 

before = df.memory_usage(deep=True).sum() / 1024**2

# Boolean — convert from the strings 'True'/'False'
df["onpromotion"] = df["onpromotion"] == "True"

# Integers — sized to actual max values
df["store_nbr"] = df["store_nbr"].astype("int8")
df["item_nbr"] = df["item_nbr"].astype("int32")
df["class"]    = df["class"].astype("int16")
df["cluster"]  = df["cluster"].astype("int8")

# Float — 7 digits of precision is ample for unit sales
df["unit_sales"] = df["unit_sales"].astype("float32")

# Categoricals — repeated text stored as integer codes + lookup
for col in ["family", "city", "state", "store_type"]:
    df[col] = df[col].astype("category")

# Drop the constant column — every row is 1, so it carries zero information
df = df.drop(columns=["perishable"], errors="ignore")

after = df.memory_usage(deep=True).sum() / 1024**2

print(f"Before: {before:.1f} MB")
print(f"After:  {after:.1f} MB")
print(f"Reduction: {(1 - after/before) * 100:.1f}%")
print("\nDtypes:\n", df.dtypes)

Before: 745.2 MB
After:  186.8 MB
Reduction: 74.9%

Dtypes:
 date           datetime64[s]
store_nbr               int8
item_nbr               int32
unit_sales           float32
onpromotion             bool
family              category
class                  int16
city                category
state               category
store_type          category
cluster                 int8
dtype: object


In [12]:
# Save the data with downcast as an intermediate step

df.to_parquet('../data/processed/protein_sales_clean.parquet', index=False)
print("Saved:", df.shape)

Saved: (5934528, 11)


In [ ]:
# Starting to build grid now below
# The data only has rows for days where something sold. No row means "sold zero," but the model can't learn from rows that don't exist. 
# Train on this as-is and it never sees a zero-demand day, so it systematically over-forecasts.

# One thing to check before we build it — a naive full cross-join would be wrong. Not every item is sold at every store (a store may never have carried a given SKU), and items get introduced or discontinued mid-timeline. 
# Filling zeros for an item a store never stocked invents fake demand history.

# 'observed=True' matters with categorical columns — it stops pandas from generating rows for combinations that never occurred. 
# The .agg(["min", "max"]) gives each pair's first and last sale date, which is how we'll bound the fill:-
# only generate zeros between a pair's first and last observed sale, not across the whole timeline.

In [13]:
n_items = df["item_nbr"].nunique()
n_stores = df["store_nbr"].nunique()
n_days = df["date"].nunique()

print(f"Items: {n_items} | Stores: {n_stores} | Days: {n_days}")
print(f"Full grid: {n_items * n_stores * n_days:,} rows")
print(f"Current:   {len(df):,} rows")
print(f"Coverage:  {len(df) / (n_items * n_stores * n_days) * 100:.1f}%")

Items: 263 | Stores: 54 | Days: 956
Full grid: 13,577,112 rows
Current:   5,934,528 rows
Coverage:  43.7%


In [14]:
# How many store-item pairs actually exist?
pairs = df.groupby(["store_nbr", "item_nbr"], observed=True).size()
print(f"\nActual store-item pairs: {len(pairs):,}")
print(f"Theoretical max: {n_items * n_stores:,}")
print(f"Pairs that exist: {len(pairs) / (n_items * n_stores) * 100:.1f}%")

# For each pair, when did it first and last sell?
pair_range = df.groupby(["store_nbr", "item_nbr"], observed=True)["date"].agg(["min", "max"])
print("\nSample of pair date ranges:")
print(pair_range.head())


Actual store-item pairs: 10,009
Theoretical max: 14,202
Pairs that exist: 70.5%

Sample of pair date ranges:
                          min        max
store_nbr item_nbr                      
1         108696   2015-01-03 2017-08-15
          108698   2015-01-02 2017-08-15
          108701   2015-01-02 2017-08-15
          108831   2015-01-08 2017-02-11
          159156   2015-01-02 2017-08-15


In [15]:
# Per-pair active window
pair_range = (
    df.groupby(["store_nbr", "item_nbr"], observed=True)["date"]
    .agg(["min", "max"])
    .reset_index()
)

# One row per pair per day within that pair's own window
pair_range["date"] = pair_range.apply(
    lambda r: pd.date_range(r["min"], r["max"], freq="D"), axis=1
)
grid = pair_range.explode("date")[["store_nbr", "item_nbr", "date"]]

grid["date"] = pd.to_datetime(grid["date"])
grid["store_nbr"] = grid["store_nbr"].astype("int8")
grid["item_nbr"] = grid["item_nbr"].astype("int32")

print(f"Grid rows: {len(grid):,}")
print(f"Actual sales rows: {len(df):,}")
print(f"Implicit zeros to add: {len(grid) - len(df):,}")

Grid rows: 7,732,111
Actual sales rows: 5,934,528
Implicit zeros to add: 1,797,583


In [16]:
# Left join: every grid row kept, sales attached where they exist
full = grid.merge(
    df[["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"]],
    on=["date", "store_nbr", "item_nbr"],
    how="left"
)

# Gaps = no sale that day = zero demand
full["unit_sales"] = full["unit_sales"].fillna(0).astype("float32")

# No sale row also means no promotion recorded
full["onpromotion"] = full["onpromotion"].fillna(False).astype(bool)

print("Shape:", full.shape)
print("Zero-sales rows:", (full["unit_sales"] == 0).sum())
print("Nulls remaining:\n", full.isna().sum())

Shape: (7732111, 5)
Zero-sales rows: 1797881
Nulls remaining:
 store_nbr      0
item_nbr       0
date           0
unit_sales     0
onpromotion    0
dtype: int64


In [17]:
items_meta = df[["item_nbr", "family", "class"]].drop_duplicates()
stores_meta = df[["store_nbr", "city", "state", "store_type", "cluster"]].drop_duplicates()

full = full.merge(items_meta, on="item_nbr", how="left")
full = full.merge(stores_meta, on="store_nbr", how="left")

print("Final shape:", full.shape)
print("Memory:", round(full.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
print("\n", full.head())

Final shape: (7732111, 11)
Memory: 184.3 MB

    store_nbr  item_nbr       date  unit_sales  onpromotion family  class  \
0          1    108696 2015-01-03         1.0        False   DELI   2636   
1          1    108696 2015-01-04         1.0        False   DELI   2636   
2          1    108696 2015-01-05         3.0        False   DELI   2636   
3          1    108696 2015-01-06         2.0        False   DELI   2636   
4          1    108696 2015-01-07         1.0        False   DELI   2636   

    city      state store_type  cluster  
0  Quito  Pichincha          D       13  
1  Quito  Pichincha          D       13  
2  Quito  Pichincha          D       13  
3  Quito  Pichincha          D       13  
4  Quito  Pichincha          D       13  


In [18]:
full.to_parquet('../data/processed/protein_grid.parquet', index=False)
print("Saved.")

Saved.
